# OurAirports references cleaning 

In [16]:
from pyspark.sql.functions import col, trim, upper

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 18, Finished, Available, Finished, False)

In [17]:
df_raw = spark.table("bronze.airports")
df_raw.printSchema()
print(f"Total rows : {df_raw.count()}")
display(df_raw.limit(5))

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 19, Finished, Available, Finished, False)

root
 |-- id: string (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: string (nullable = true)
 |-- longitude_deg: string (nullable = true)
 |-- elevation_ft: string (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

Total rows : 85835


SynapseWidget(Synapse.DataFrame, 5d539c85-5606-48ef-a76e-535e19d72e49)

In [18]:
df_eu = df_raw.filter(col("continent") == "EU")

print(f"European airports total : {df_eu.count()}")

# Null rates on all code columns
from pyspark.sql.functions import count, when

display(
    df_eu.select([
        count(when(col(c).isNull() | (col(c) == ""), 1)).alias(c)
        for c in ["icao_code", "gps_code", "local_code", "iata_code", "ident", "keywords"]
    ])
)

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 20, Finished, Available, Finished, False)

European airports total : 12531


SynapseWidget(Synapse.DataFrame, 395d9f13-735e-4cdc-9c70-d6e8541e3d35)

In [19]:
# Rows where icao_code is NULL but gps_code is not
df_icao_null_gps_ok = df_eu.filter(
    (col("icao_code").isNull() | (col("icao_code") == "")) &
    col("gps_code").isNotNull() & (col("gps_code") != "")
)
print(f"icao_code NULL but gps_code present : {df_icao_null_gps_ok.count()}")
display(df_icao_null_gps_ok.select("ident", "icao_code", "gps_code", "local_code", "iata_code", "name", "type", "keywords").limit(20))

# Rows where both icao_code and gps_code are NULL
df_both_null = df_eu.filter(
    (col("icao_code").isNull() | (col("icao_code") == "")) &
    (col("gps_code").isNull() | (col("gps_code") == ""))
)
print(f"Both icao_code and gps_code NULL : {df_both_null.count()}")
display(df_both_null.select("ident", "icao_code", "gps_code", "local_code", "iata_code", "name", "type", "keywords").limit(20))

# Check: when both are present, do they match ?
df_both_present = df_eu.filter(
    col("icao_code").isNotNull() & (col("icao_code") != "") &
    col("gps_code").isNotNull() & (col("gps_code") != "")
)
mismatch = df_both_present.filter(col("icao_code") != col("gps_code")).count()
print(f"icao_code ≠ gps_code (when both present) : {mismatch}")

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 21, Finished, Available, Finished, False)

icao_code NULL but gps_code present : 2011


SynapseWidget(Synapse.DataFrame, 0268d1b5-1467-4475-8ce9-fbc44471b688)

Both icao_code and gps_code NULL : 8439


SynapseWidget(Synapse.DataFrame, 334b44f3-ca81-42cd-aebd-4ac9979ace46)

icao_code ≠ gps_code (when both present) : 3


In [20]:
from pyspark.sql.functions import col, trim, upper, coalesce

EXCLUDED_TYPES = ["heliport", "seaplane_base", "closed", "balloonport"]

df_clean = (
    df_raw
    .filter(col("continent") == "EU")
    .filter(~col("type").isin(EXCLUDED_TYPES))
    .withColumn(
        "airport_icao",
        trim(upper(coalesce(col("icao_code"), col("gps_code"))))
    )
    .filter(col("airport_icao").isNotNull() & (col("airport_icao") != ""))
    .select(
        col("airport_icao"),
        trim(upper(col("iata_code"))).alias("airport_iata"),
        col("name").alias("airport_name"),
        col("type").alias("airport_type"),
        col("municipality").alias("city"),
        col("iso_country").alias("country"),
        col("latitude_deg").cast("double").alias("latitude"),
        col("longitude_deg").cast("double").alias("longitude"),
        col("icao_code").isNotNull().alias("icao_is_official")
    )
)

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 22, Finished, Available, Finished, False)

In [21]:
from pyspark.sql.functions import count, when

print(f"Raw EU total             : {df_raw.filter(col('continent') == 'EU').count()}")
print(f"After type filter + code : {df_clean.count()}")

# Check duplicates on airport_icao
dup_df = df_clean.groupBy("airport_icao").count().filter(col("count") > 1)
print(f"Duplicate airport_icao   : {dup_df.count()}")

# Inspect the duplicates — understand WHY they exist before deciding
display(
    df_clean.join(dup_df.select("airport_icao"), on="airport_icao", how="inner")
    .orderBy("airport_icao")
)

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 23, Finished, Available, Finished, False)

Raw EU total             : 12531
After type filter + code : 3186
Duplicate airport_icao   : 0


SynapseWidget(Synapse.DataFrame, 3e484ace-6b91-445a-aa25-293d4ff4f84d)

### SCD Type 2 handler

In [22]:
from pyspark.sql.functions import md5, concat_ws, current_date, lit
from pyspark.sql import functions as F

# Columns that define a "change" (structural/naming attributes only)
SCD_COLS = ["airport_name", "airport_iata", "airport_type", "city", "country", "latitude", "longitude"]

# Add hash to detect changes
df_incoming = df_clean.withColumn(
    "scd_hash",
    md5(concat_ws("|", *[col(c).cast("string") for c in SCD_COLS]))
)

TARGET = "silver.airport"

if not spark.catalog.tableExists(TARGET):
    # ── First load ──────────────────────────────────────────────
    df_output = (
        df_incoming
        .withColumn("valid_from",  current_date())
        .withColumn("valid_to",    lit(None).cast("date"))
        .withColumn("is_current",  lit(True))
    )
    df_output.write.mode("overwrite").saveAsTable(TARGET)
    print(f"First load: {df_output.count()} rows → {TARGET}")

else:
    # ── Incremental SCD2 ────────────────────────────────────────
    df_existing = spark.table(TARGET)
    df_current  = df_existing.filter(col("is_current") == True)
    df_history  = df_existing.filter(col("is_current") == False)

    # Unchanged: in current AND hash matches incoming
    df_unchanged = (
        df_current.alias("cur")
        .join(df_incoming.select("airport_icao", "scd_hash").alias("inc"),
              on="airport_icao", how="inner")
        .filter(col("cur.scd_hash") == col("inc.scd_hash"))
        .select("cur.*")
    )

    # Expired: in current AND (hash changed OR disappeared from incoming)
    df_expired = (
        df_current.alias("cur")
        .join(df_incoming.select("airport_icao", "scd_hash").alias("inc"),
              on="airport_icao", how="left")
        .filter(
            col("inc.airport_icao").isNull() |                  
            (col("cur.scd_hash") != col("inc.scd_hash"))        
        )
        .select("cur.*")
        .withColumn("valid_to",   current_date())
        .withColumn("is_current", lit(False))
    )

# New versions: new airports OR changed airports (insert fresh row)
    df_new_versions = (
        df_incoming.alias("inc")
        .join(df_current.select("airport_icao", "scd_hash").alias("cur"),
              on="airport_icao", how="left")
        .filter(
            col("cur.airport_icao").isNull() |                  
            (col("inc.scd_hash") != col("cur.scd_hash"))      
        )
        .select("inc.*")
        .withColumn("valid_from",  current_date())
        .withColumn("valid_to",    lit(None).cast("date"))
        .withColumn("is_current",  lit(True))
    )

    df_output = (
        df_history
        .unionByName(df_unchanged)
        .unionByName(df_expired)
        .unionByName(df_new_versions)
    )

    df_output.write.mode("overwrite").saveAsTable(TARGET)

    print(f"Unchanged     : {df_unchanged.count()}")
    print(f"Expired       : {df_expired.count()}")
    print(f"New versions  : {df_new_versions.count()}")
    print(f"Total written : {spark.table(TARGET).count()}")

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 24, Finished, Available, Finished, False)

Unchanged     : 3186
Expired       : 0
New versions  : 0
Total written : 3186


In [23]:
df_result = spark.table("silver.airport")

print(f"Total rows    : {df_result.count()}")
print(f"is_current    : {df_result.filter(col('is_current') == True).count()}")
print(f"is_historical : {df_result.filter(col('is_current') == False).count()}")

display(df_result.filter(col("is_current") == True).limit(10))

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 25, Finished, Available, Finished, False)

Total rows    : 3186
is_current    : 3186
is_historical : 0


SynapseWidget(Synapse.DataFrame, 62d26839-ec51-4c88-824f-8b1e26ea4224)

### Validation checks

In [26]:
df_silver = spark.table("silver.airline")
errors = []

# CRITICAL — primary key uniqueness
dupes = df_silver.groupBy("icao_code").count().filter(col("count") > 1).count()
if dupes > 0:
    errors.append(f"FAIL: {dupes} duplicate icao_code")

# CRITICAL — no null primary key
nulls = df_silver.filter(col("icao_code").isNull()).count()
if nulls > 0:
    errors.append(f"FAIL: {nulls} null icao_code")

# CRITICAL — carrier_type values must be within expected set
valid_types = {"LCC", "Network", "Regional", "Other"}
invalid = df_silver.filter(~col("carrier_type").isin(valid_types)).count()
if invalid > 0:
    errors.append(f"FAIL: {invalid} invalid carrier_type values")

# INFO — carrier_type distribution
display(df_silver.groupBy("carrier_type").count().orderBy("count", ascending=False))

if errors:
    raise ValueError("\n".join(errors))
else:
    print(f"All checks passed — {df_silver.count()} rows")

StatementMeta(, 941f6007-70c2-4845-9da7-6a0da0647ff5, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee7d8ea7-3eb0-4fe3-8ecf-3d2aac199eb4)

All checks passed — 869 rows
